In [12]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from keras.layers import SimpleRNN, Embedding, Dense, LSTM
from keras.models import Sequential
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from sklearn.utils.class_weight import compute_class_weight

In [4]:
#Set random seed for reproducability
np.random.seed(42)

# Question 1

## Question 1.1

In [6]:
#Read JSON File
df = pd.read_json("Datasets/research_abstracts.json")

In [8]:
#Check missing values
df.isnull().sum()

DOI         4
Abstract    0
Label       0
dtype: int64

In [10]:
#Check duplicates
df.duplicated().sum()

2

In [12]:
#Drop missing values
df = df.dropna()

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1134 entries, 0 to 1137
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   DOI       1134 non-null   object
 1   Abstract  1134 non-null   object
 2   Label     1134 non-null   object
dtypes: object(3)
memory usage: 35.4+ KB


In [16]:
df.head()

,DOI,Abstract,Label
0,10.1155/2021/1071145,Forward-looking forecasting of the inflation r...,ai
1,10.1007/s43546-022-00384-2,Our study examined the disaggregation of infla...,ai
2,10.1002/for.2667,This paper develops a dynamic factor model tha...,classical
3,10.1007/s42973-021-00091-x,"In this paper, I use a simple SIR Macro model ...",classical
4,10.1007/s40258-012-0003-z,BackgroundGame theory is useful for identifyin...,classical


In [18]:
nltk.download('punkt_tab')
stop_words = set(stopwords.words("english"))

#Preprocess function
def clean_text(text):
    #Remove punctuation, special characters, numbers
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    #Lowercase
    text = text.lower()

    #Tokenize
    tokens = word_tokenize(text)

    #Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    text = " ".join(tokens)
    
    return text

[nltk_data] Downloading package punkt_tab to C:\Users\Jean
[nltk_data]     Kotze\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [20]:
df["Clean Abstract"] = df["Abstract"].apply(clean_text)

In [22]:
#Clean dataset
clean_df = df[["DOI", "Clean Abstract", "Label"]].copy()

In [24]:
clean_df.head()

,DOI,Clean Abstract,Label
0,10.1155/2021/1071145,forward looking forecasting inflation rate cou...,ai
1,10.1007/s43546-022-00384-2,study examined disaggregation inflation compon...,ai
2,10.1002/for.2667,paper develops dynamic factor model uses euro ...,classical
3,10.1007/s42973-021-00091-x,paper use simple sir macro model examine japan...,classical
4,10.1007/s40258-012-0003-z,backgroundgame theory useful identifying condi...,classical


## Question 1.2

DOI will be excluded from the table as it acts as an identifier for the research papers, but doesn’t contain any information that can be used for classification purposes.

In [26]:
clean_df = clean_df.drop("DOI", axis=1)

In [28]:
clean_df.head()

,Clean Abstract,Label
0,forward looking forecasting inflation rate cou...,ai
1,study examined disaggregation inflation compon...,ai
2,paper develops dynamic factor model uses euro ...,classical
3,paper use simple sir macro model examine japan...,classical
4,backgroundgame theory useful identifying condi...,classical


Assumption: The label column would be the target variable for classification.

TD-IDF Feature selection will be chosen as the feature modelling technique.

In [30]:
X = clean_df["Clean Abstract"]
y = clean_df["Label"]

tfidf = TfidfVectorizer()
#Train test split 80-20
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [32]:
#Feature selection on train and test set
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

## Question 1.3

For the baseline model Naive Bayes will be used.

In [34]:
#Train baseline model
baseline_model = MultinomialNB()
baseline_model.fit(X_train_tfidf, y_train)

MultinomialNB()

In [36]:
y_train_pred = baseline_model.predict(X_train_tfidf)
#Train performance
print("NB Training Model Performance")
print("Accuracy:", accuracy_score(y_train, y_train_pred))
print(classification_report(y_train, y_train_pred))

NB Training Model Performance
Accuracy: 0.8114663726571113
              precision    recall  f1-score   support

          ai       1.00      0.09      0.16       187
   classical       0.81      1.00      0.89       720

    accuracy                           0.81       907
   macro avg       0.90      0.54      0.53       907
weighted avg       0.85      0.81      0.74       907



In [38]:
y_test_pred = baseline_model.predict(X_test_tfidf)
#Test performance
print("NB Testing Model Performance")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred))

NB Testing Model Performance
Accuracy: 0.801762114537445
              precision    recall  f1-score   support

          ai       1.00      0.04      0.08        47
   classical       0.80      1.00      0.89       180

    accuracy                           0.80       227
   macro avg       0.90      0.52      0.49       227
weighted avg       0.84      0.80      0.72       227



The Naïve Bayes classifier performs well in training and testing as it gives 81.15% accuracy in training and 80.18% accuracy in testing. It tends to struggle for both splits in classifying abstracts into the AI category, since the recall and f1 scores are low, which might be due to class imbalances, since there are more classical abstracts than AI abstracts in the dataset.

# Question 2

## Question 2.1

For the next models, LR, SVM, RNN and LSTM will be used

### i) Logistic Regression

In [40]:
#Initialise model
lr_model = LogisticRegression(max_iter=1000)

#Train model
lr_model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [42]:
y_train_pred = lr_model.predict(X_train_tfidf)
#Train performance
print("LR Training Model Performance")
print("Accuracy:", accuracy_score(y_train, y_train_pred))
print(classification_report(y_train, y_train_pred))

LR Training Model Performance
Accuracy: 0.8853362734288864
              precision    recall  f1-score   support

          ai       1.00      0.44      0.61       187
   classical       0.87      1.00      0.93       720

    accuracy                           0.89       907
   macro avg       0.94      0.72      0.77       907
weighted avg       0.90      0.89      0.87       907



In [44]:
y_test_pred = lr_model.predict(X_test_tfidf)
#Test performance
print("LR Testing Model Performance")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred))

LR Testing Model Performance
Accuracy: 0.8458149779735683
              precision    recall  f1-score   support

          ai       1.00      0.26      0.41        47
   classical       0.84      1.00      0.91       180

    accuracy                           0.85       227
   macro avg       0.92      0.63      0.66       227
weighted avg       0.87      0.85      0.81       227



## ii) Support Vector Machines

In [46]:
#Initialise model
svm_model = LinearSVC(C=1.0)

#Train model
svm_model.fit(X_train_tfidf, y_train)

LinearSVC()

In [48]:
y_train_pred = svm_model.predict(X_train_tfidf)
#Train performance
print("SVM Training Model Performance")
print("Accuracy:", accuracy_score(y_train, y_train_pred))
print(classification_report(y_train, y_train_pred))

SVM Training Model Performance
Accuracy: 1.0
              precision    recall  f1-score   support

          ai       1.00      1.00      1.00       187
   classical       1.00      1.00      1.00       720

    accuracy                           1.00       907
   macro avg       1.00      1.00      1.00       907
weighted avg       1.00      1.00      1.00       907



In [50]:
y_test_pred = svm_model.predict(X_test_tfidf)
#Test performance
print("SVM Testing Model Performance")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred))

SVM Testing Model Performance
Accuracy: 0.9251101321585903
              precision    recall  f1-score   support

          ai       0.94      0.68      0.79        47
   classical       0.92      0.99      0.95       180

    accuracy                           0.93       227
   macro avg       0.93      0.83      0.87       227
weighted avg       0.93      0.93      0.92       227



## iii) Recurrent Neural Network

For RNN and LTSM we need to make sure the data is in the correct format.

In [52]:
rnn_df = clean_df

In [54]:
rnn_df.head()

,Clean Abstract,Label
0,forward looking forecasting inflation rate cou...,ai
1,study examined disaggregation inflation compon...,ai
2,paper develops dynamic factor model uses euro ...,classical
3,paper use simple sir macro model examine japan...,classical
4,backgroundgame theory useful identifying condi...,classical


In [56]:
abstracts = []
labels = []

for i, label in enumerate(rnn_df["Label"]):
    abstracts.append(rnn_df["Clean Abstract"].iloc[i])

    if label == "classical":
        labels.append(0)
    else:
        labels.append(1)

abstracts = np.asarray(abstracts)
labels = np.asarray(labels)

print(len(abstracts))
print(len(labels))

1134
1134


In [58]:
max_features = 10000
maxlen = 500

#Split the data
X_train_rnn, X_test_rnn, y_train_rnn, y_test_rnn = train_test_split(
    abstracts,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

#Tokenize
tokenizer = Tokenizer(num_words=max_features)
tokenizer.fit_on_texts(X_train_rnn)

#Convert abstracts into sequences
X_train_sequences = tokenizer.texts_to_sequences(X_train_rnn)
X_test_sequences = tokenizer.texts_to_sequences(X_test_rnn)

#Pad sequences
X_train_pad = pad_sequences(X_train_sequences, maxlen=maxlen)
X_test_pad = pad_sequences(X_test_sequences, maxlen=maxlen)

print("Unique words:", len(tokenizer.word_index))
print("Training shape:", X_train_pad.shape)
print("Testing shape:", X_test_pad.shape)

Unique words: 8646
Training shape: (907, 500)
Testing shape: (227, 500)


In [60]:
model = Sequential()

model.add(Embedding(max_features, 32))
model.add(SimpleRNN(32))
model.add(Dense(1, activation="sigmoid"))

model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn = model.fit(
    X_train_pad,
    y_train_rnn,
    epochs=10,
    batch_size=60,
    validation_split=0.2
)

Epoch 1/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 6s 121ms/step - accuracy: 0.5788 - loss: 0.6753 - val_accuracy: 0.7912 - val_loss: 0.5851
Epoch 2/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 91ms/step - accuracy: 0.8069 - loss: 0.5099 - val_accuracy: 0.7582 - val_loss: 0.5931
Epoch 3/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 100ms/step - accuracy: 0.8880 - loss: 0.4219 - val_accuracy: 0.7912 - val_loss: 0.5349
Epoch 4/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 95ms/step - accuracy: 0.9579 - loss: 0.2923 - val_accuracy: 0.7967 - val_loss: 0.5144
Epoch 5/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 117ms/step - accuracy: 0.9974 - loss: 0.1759 - val_accuracy: 0.8132 - val_loss: 0.4701
Epoch 6/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 94ms/step - accuracy: 0.9839 - loss: 0.1483 - val_accuracy: 0.8132 - val_loss: 0.4740
Epoch 7/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 91ms/step - accuracy: 0.9979 - loss: 0.0752 - val_accuracy: 0.8077 - val_loss: 0.4773
Epoch 8/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 107ms/step - accuracy: 0.9995 - loss: 0.0915 - val_accuracy: 0.7912

In [62]:
y_train_pred = model.predict(X_train_pad)
y_train_pred_classes = (y_train_pred >= 0.5).astype(int).flatten()
#Train performance
print("RNN Training Model Performance")
print("Accuracy:", accuracy_score(y_train_rnn, y_train_pred_classes))
print(classification_report(y_train_rnn, y_train_pred_classes))

29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step
RNN Training Model Performance
Accuracy: 0.9349503858875413
              precision    recall  f1-score   support

           0       0.97      0.95      0.96       720
           1       0.82      0.87      0.85       187

    accuracy                           0.93       907
   macro avg       0.89      0.91      0.90       907
weighted avg       0.94      0.93      0.94       907



In [64]:
y_test_pred = model.predict(X_test_pad)
y_test_pred_classes = (y_test_pred >= 0.5).astype(int).flatten()
#Test performance
print("RNN Test Model Performance")
print("Accuracy:", accuracy_score(y_test_rnn, y_test_pred_classes))
print(classification_report(y_test_rnn, y_test_pred_classes))

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
RNN Test Model Performance
Accuracy: 0.7709251101321586
              precision    recall  f1-score   support

           0       0.88      0.83      0.85       180
           1       0.46      0.55      0.50        47

    accuracy                           0.77       227
   macro avg       0.67      0.69      0.68       227
weighted avg       0.79      0.77      0.78       227



### iv) LSTM

In [66]:
model_lstm = Sequential()

model_lstm.add(Embedding(max_features, 32))
model_lstm.add(LSTM(128, dropout=0.2, recurrent_dropout = 0.2))
model_lstm.add(Dense(1, activation="sigmoid"))

model_lstm.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm = model_lstm.fit(
    X_train_pad,
    y_train_rnn,
    epochs=10,
    batch_size=60,
    validation_split=0.2
)

Epoch 1/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 11s 557ms/step - accuracy: 0.7148 - loss: 0.6401 - val_accuracy: 0.7912 - val_loss: 0.7324
Epoch 2/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 620ms/step - accuracy: 0.7949 - loss: 0.5525 - val_accuracy: 0.7912 - val_loss: 0.5624
Epoch 3/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 634ms/step - accuracy: 0.8009 - loss: 0.5141 - val_accuracy: 0.7912 - val_loss: 0.5244
Epoch 4/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 9s 704ms/step - accuracy: 0.8048 - loss: 0.5003 - val_accuracy: 0.7912 - val_loss: 0.5330
Epoch 5/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 609ms/step - accuracy: 0.7789 - loss: 0.5253 - val_accuracy: 0.7912 - val_loss: 0.4943
Epoch 6/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 617ms/step - accuracy: 0.8066 - loss: 0.4560 - val_accuracy: 0.7912 - val_loss: 0.4510
Epoch 7/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 604ms/step - accuracy: 0.8447 - loss: 0.3703 - val_accuracy: 0.7802 - val_loss: 0.4951
Epoch 8/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 609ms/step - accuracy: 0.8315 - loss: 0.3872 - val_accuracy: 0

In [70]:
y_train_pred = model_lstm.predict(X_train_pad)
y_train_pred_classes = (y_train_pred >= 0.5).astype(int).flatten()
# Training performance
print("LSTM Training Model Performance")
print("Accuracy:", accuracy_score(y_train_rnn, y_train_pred_classes))
print(classification_report(y_train_rnn, y_train_pred_classes))

29/29 ━━━━━━━━━━━━━━━━━━━━ 3s 83ms/step
LSTM Training Model Performance
Accuracy: 0.9415656008820287
              precision    recall  f1-score   support

           0       0.94      0.99      0.96       720
           1       0.97      0.74      0.84       187

    accuracy                           0.94       907
   macro avg       0.95      0.87      0.90       907
weighted avg       0.94      0.94      0.94       907



In [72]:
y_test_pred = model_lstm.predict(X_test_pad)
y_test_pred_classes = (y_test_pred >= 0.5).astype(int).flatten()
#Test performance
print("LSTM Test Model Performance")
print("Accuracy:", accuracy_score(y_test_rnn, y_test_pred_classes))
print(classification_report(y_test_rnn, y_test_pred_classes))

8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/step
LSTM Test Model Performance
Accuracy: 0.8502202643171806
              precision    recall  f1-score   support

           0       0.86      0.96      0.91       180
           1       0.74      0.43      0.54        47

    accuracy                           0.85       227
   macro avg       0.80      0.69      0.73       227
weighted avg       0.84      0.85      0.83       227



## Question 2.2

In [74]:
clean_df["Label"].value_counts()

Label
classical    900
ai           234
Name: count, dtype: int64

Classes are imbalanced, thus class balancing can improve performance.

### i) Logistic Regression

In [76]:
#Initialise model
lr_model_b = LogisticRegression(max_iter=1000, class_weight="balanced")

#Train model
lr_model_b.fit(X_train_tfidf, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [78]:
y_train_pred = lr_model_b.predict(X_train_tfidf)
#Train performance
print("LR Balanced Training Model Performance")
print("Accuracy:", accuracy_score(y_train, y_train_pred))
print(classification_report(y_train, y_train_pred))

LR Balanced Training Model Performance
Accuracy: 0.9790518191841234
              precision    recall  f1-score   support

          ai       0.91      0.99      0.95       187
   classical       1.00      0.97      0.99       720

    accuracy                           0.98       907
   macro avg       0.96      0.98      0.97       907
weighted avg       0.98      0.98      0.98       907



In [80]:
y_test_pred = lr_model_b.predict(X_test_tfidf)
#Test performance
print("LR Balanced Testing Model Performance")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred))

LR Balanced Testing Model Performance
Accuracy: 0.8766519823788547
              precision    recall  f1-score   support

          ai       0.69      0.72      0.71        47
   classical       0.93      0.92      0.92       180

    accuracy                           0.88       227
   macro avg       0.81      0.82      0.82       227
weighted avg       0.88      0.88      0.88       227



### ii) Support Vector Machines

In [82]:
#Initialise model
svm_model_b = LinearSVC(C=1.0, class_weight="balanced")

#Train model
svm_model_b.fit(X_train_tfidf, y_train)

LinearSVC(class_weight='balanced')

In [84]:
y_train_pred = svm_model_b.predict(X_train_tfidf)
#Train performance
print("SVM Balanced Training Model Performance")
print("Accuracy:", accuracy_score(y_train, y_train_pred))
print(classification_report(y_train, y_train_pred))

SVM Balanced Training Model Performance
Accuracy: 1.0
              precision    recall  f1-score   support

          ai       1.00      1.00      1.00       187
   classical       1.00      1.00      1.00       720

    accuracy                           1.00       907
   macro avg       1.00      1.00      1.00       907
weighted avg       1.00      1.00      1.00       907



In [86]:
y_test_pred = svm_model_b.predict(X_test_tfidf)
#Test performance
print("SVM Balanced Testing Model Performance")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred))

SVM Balanced Testing Model Performance
Accuracy: 0.9074889867841409
              precision    recall  f1-score   support

          ai       0.82      0.70      0.76        47
   classical       0.93      0.96      0.94       180

    accuracy                           0.91       227
   macro avg       0.88      0.83      0.85       227
weighted avg       0.90      0.91      0.90       227



Compute class weights for RNN and LSTM

In [90]:
classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))

## iii) RNN

In [92]:
model = Sequential()

model.add(Embedding(max_features, 32))
model.add(SimpleRNN(32))
model.add(Dense(1, activation="sigmoid"))

model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_b = model.fit(
    X_train_pad,
    y_train_rnn,
    epochs=10,
    batch_size=60,
    validation_split=0.2,
    class_weight=class_weights
)

Epoch 1/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 116ms/step - accuracy: 0.6290 - loss: 0.6538 - val_accuracy: 0.7912 - val_loss: 0.5589
Epoch 2/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 90ms/step - accuracy: 0.7944 - loss: 0.5323 - val_accuracy: 0.7912 - val_loss: 0.5519
Epoch 3/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 93ms/step - accuracy: 0.8393 - loss: 0.4298 - val_accuracy: 0.7912 - val_loss: 0.5318
Epoch 4/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 94ms/step - accuracy: 0.9769 - loss: 0.2962 - val_accuracy: 0.7802 - val_loss: 0.5371
Epoch 5/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 91ms/step - accuracy: 0.9970 - loss: 0.1902 - val_accuracy: 0.7692 - val_loss: 0.5266
Epoch 6/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 102ms/step - accuracy: 0.9961 - loss: 0.1565 - val_accuracy: 0.6538 - val_loss: 0.6091
Epoch 7/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 94ms/step - accuracy: 0.9997 - loss: 0.1241 - val_accuracy: 0.7198 - val_loss: 0.5631
Epoch 8/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 109ms/step - accuracy: 0.9986 - loss: 0.0654 - val_accuracy: 0.6484 

In [94]:
y_train_pred = model.predict(X_train_pad)
y_train_pred_classes = (y_train_pred >= 0.5).astype(int).flatten()
#Train performance
print("RNN Balanced Training Model Performance")
print("Accuracy:", accuracy_score(y_train_rnn, y_train_pred_classes))
print(classification_report(y_train_rnn, y_train_pred_classes))

29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step
RNN Balanced Training Model Performance
Accuracy: 0.9360529217199559
              precision    recall  f1-score   support

           0       0.97      0.95      0.96       720
           1       0.82      0.88      0.85       187

    accuracy                           0.94       907
   macro avg       0.90      0.91      0.90       907
weighted avg       0.94      0.94      0.94       907



In [98]:
y_test_pred = model.predict(X_test_pad)
y_test_pred_classes = (y_test_pred >= 0.5).astype(int).flatten()
#Test performance
print("RNN Balanced Test Model Performance")
print("Accuracy:", accuracy_score(y_test_rnn, y_test_pred_classes))
print(classification_report(y_test_rnn, y_test_pred_classes))

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
RNN Balanced Test Model Performance
Accuracy: 0.6696035242290749
              precision    recall  f1-score   support

           0       0.85      0.71      0.77       180
           1       0.32      0.51      0.39        47

    accuracy                           0.67       227
   macro avg       0.58      0.61      0.58       227
weighted avg       0.74      0.67      0.69       227



## iv) LSTM

In [100]:
model_lstm = Sequential()

model_lstm.add(Embedding(max_features, 32))
model_lstm.add(LSTM(128, dropout=0.2, recurrent_dropout = 0.2))
model_lstm.add(Dense(1, activation="sigmoid"))

model_lstm.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_b = model_lstm.fit(
    X_train_pad,
    y_train_rnn,
    epochs=10,
    batch_size=60,
    validation_split=0.2,
    class_weight=class_weights
)

Epoch 1/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 12s 635ms/step - accuracy: 0.7568 - loss: 0.6333 - val_accuracy: 0.7912 - val_loss: 0.5088
Epoch 2/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 9s 662ms/step - accuracy: 0.8003 - loss: 0.5249 - val_accuracy: 0.7912 - val_loss: 0.5046
Epoch 3/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 646ms/step - accuracy: 0.7749 - loss: 0.5274 - val_accuracy: 0.7912 - val_loss: 0.5031
Epoch 4/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 616ms/step - accuracy: 0.7795 - loss: 0.5012 - val_accuracy: 0.7912 - val_loss: 0.4988
Epoch 5/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 606ms/step - accuracy: 0.8259 - loss: 0.4124 - val_accuracy: 0.8077 - val_loss: 0.4757
Epoch 6/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 628ms/step - accuracy: 0.8784 - loss: 0.3190 - val_accuracy: 0.8132 - val_loss: 0.4527
Epoch 7/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 609ms/step - accuracy: 0.9105 - loss: 0.2860 - val_accuracy: 0.8132 - val_loss: 0.4537
Epoch 8/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 636ms/step - accuracy: 0.9358 - loss: 0.2358 - val_accuracy: 0

In [102]:
y_train_pred = model_lstm.predict(X_train_pad)
y_train_pred_classes = (y_train_pred >= 0.5).astype(int).flatten()
# Training performance
print("LSTM Balanced Training Model Performance")
print("Accuracy:", accuracy_score(y_train_rnn, y_train_pred_classes))
print(classification_report(y_train_rnn, y_train_pred_classes))

29/29 ━━━━━━━━━━━━━━━━━━━━ 3s 88ms/step
LSTM Balanced Training Model Performance
Accuracy: 0.960308710033076
              precision    recall  f1-score   support

           0       0.97      0.98      0.98       720
           1       0.94      0.87      0.90       187

    accuracy                           0.96       907
   macro avg       0.95      0.93      0.94       907
weighted avg       0.96      0.96      0.96       907



In [104]:
y_test_pred = model_lstm.predict(X_test_pad)
y_test_pred_classes = (y_test_pred >= 0.5).astype(int).flatten()
#Test performance
print("LSTM Balanced Test Model Performance")
print("Accuracy:", accuracy_score(y_test_rnn, y_test_pred_classes))
print(classification_report(y_test_rnn, y_test_pred_classes))

8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step
LSTM Balanced Test Model Performance
Accuracy: 0.8590308370044053
              precision    recall  f1-score   support

           0       0.90      0.92      0.91       180
           1       0.67      0.62      0.64        47

    accuracy                           0.86       227
   macro avg       0.79      0.77      0.78       227
weighted avg       0.86      0.86      0.86       227



The highest test accuracy is SVM classification with no class balance